In [ ]:

# %%
"""kaggle_dataset_analysis_colab.py
Colab-adaptable script / notebook for analyzing a Kaggle dataset from a Kaggle dataset URL.
Usage in Colab:
1. Upload your kaggle.json (from your Kaggle account) to /root/.kaggle/kaggle.json or use the snippets below to authenticate.
2. Set the DATASET_URL (Kaggle dataset URL) or directly set DATASET_REF (owner/dataset-name).
3. Run cells sequentially.

This script is written as a sequence of cells (marked by "# %%") so you can paste it into a .py file and open it as a notebook (or run as script after minor edits).
"""
# %%
# --------------------
# 0. Configuration
# --------------------
# Change these variables if needed before executing
DATASET_URL = None  # Example: "https://www.kaggle.com/uciml/iris"
# If you prefer, fill dataset_ref directly as "<owner>/<dataset-name>"
DATASET_REF = None  # e.g. "uciml/iris"

# Target detection: Leave as None to let the script auto-detect a target column (best-effort).
TARGET_COLUMN = None  # set to the exact column name if you want to force a target

# If you want to load a specific CSV file from the dataset (by filename), set it here.
FORCE_CSV_FILENAME = None  # e.g. "data.csv"

# Random seed for reproducibility
RANDOM_STATE = 42

# %%
# 1. Install required packages (uncomment when running in Colab)
# In Colab run these once:
# !pip install kaggle category_encoders xgboost lightgbm openpyxl

# %%
# 2. Kaggle authentication & download dataset
import os
import zipfile
import glob
import shutil
from pathlib import Path

def kaggle_download(dataset_url= "https://www.kaggle.com/datasets/hugomathien/soccer/data", dataset_ref=None, dest_dir='dataset', force=False):
    """
    Downloads a Kaggle dataset archive to dest_dir and extracts it.
    Requires kaggle API credentials in ~/.kaggle/kaggle.json or environment variables KAGGLE_USERNAME/KAGGLE_KEY.
    If you run on Colab, upload kaggle.json to /root/.kaggle/kaggle.json
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except Exception as e:
        raise RuntimeError("kaggle package not found. Install with: pip install kaggle") from e

    api = KaggleApi()
    api.authenticate()

    if dataset_ref is None:
        if dataset_url is None:
            raise ValueError("Provide dataset_url or dataset_ref")
        # try to extract owner/dataset from URL
        # typical formats: https://www.kaggle.com/owner/dataset or https://kaggle.com/owner/dataset
        parts = dataset_url.rstrip('/').split('/')
        if len(parts) < 2:
            raise ValueError("Could not parse dataset_url")
        dataset_ref = '/'.join(parts[-2:])

    os.makedirs(dest_dir, exist_ok=True)
    print(f"Downloading dataset {dataset_ref} to {dest_dir} ...")
    api.dataset_download_files(dataset_ref, path=dest_dir, unzip=True, quiet=False)
    print("Download/extract complete.")
    return dest_dir

# Only run kaggle_download if the user provided dataset info
if DATASET_REF is None and DATASET_URL is None:
    print("No Kaggle dataset specified. Please set DATASET_URL or DATASET_REF and re-run the download cell.")
else:
    # In Colab, ensure kaggle.json is in /root/.kaggle/kaggle.json and perm 600
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists():
        print("Warning: ~/.kaggle/kaggle.json not found. Make sure to upload it or set KAGGLE_USERNAME/KAGGLE_KEY env vars in Colab.")
    # Attempt download (comment out if you prefer to upload files manually)
    try:
        kaggle_download(dataset_url=DATASET_URL, dataset_ref=DATASET_REF, dest_dir='dataset')
    except Exception as e:
        print("Dataset download failed (often due to missing kaggle credentials in this environment).")
        print(str(e))

# %%
# 3. Locate a CSV file to analyze
import pandas as pd
data_files = glob.glob('dataset/**/*.csv', recursive=True) + glob.glob('dataset/*.csv')
data_files = sorted(list(set(data_files)))
print("Found CSV files:", data_files)

if not data_files:
    print("No CSV files detected in 'dataset' folder. Please upload your CSV into the workspace or re-check the Kaggle dataset contents.")
else:
    # choose a CSV file
    if FORCE_CSV_FILENAME:
        csv_path = FORCE_CSV_FILENAME
    else:
        csv_path = data_files[0]
    print("Loading", csv_path)
    df = pd.read_csv(csv_path)
    print("Shape:", df.shape)
    display(df.head())

# %%
# 4. Basic metadata & automatic target detection (best-effort)
import numpy as np
def detect_target(df, forced_target=None):
    if forced_target and forced_target in df.columns:
        return forced_target, 'user-specified'
    # common candidate names
    candidates = ['target', 'label', 'class', 'y', 'outcome', 'survived']
    for c in candidates:
        if c in df.columns:
            return c, f'found common name "{c}"'
    # try last column if it's not obviously an ID or text column
    col = df.columns[-1]
    if df[col].dtype in [np.number, 'int64', 'float64'] or df[col].nunique() < df.shape[0] * 0.5:
        return col, 'fallback: last column'
    # otherwise choose numeric column with few unique values
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        # choose the one with least missing and reasonable cardinality
        numeric_cols = sorted(numeric_cols, key=lambda c: (df[c].isna().sum(), df[c].nunique()))
        return numeric_cols[0], 'fallback: numeric column heuristic'
    # else choose last column anyway
    return df.columns[-1], 'fallback: last column'

if 'df' in globals():
    target_col, reason = detect_target(df, TARGET_COLUMN)
    print("Auto-detected target:", target_col, "| Reason:", reason)
    print("Target dtype:", df[target_col].dtype if target_col in df.columns else "N/A")

# %%
# 5. Preprocessing pipeline
# Steps implemented:
# - Duplicate handling
# - Basic formatting
# - Missing value imputation: IterativeImputer for numerics; most_frequent for categoricals; optional KNN or model-based strategies
# - Encoding: OneHot for low-cardinality categoricals, TargetEncoder for high-cardinality (category_encoders)
# - Scaling: StandardScaler for numeric features
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import category_encoders as ce

def preprocess_dataframe(df, target_col, max_onehot_cardinality=20, use_iterative_imputer=True):
    df = df.copy()
    # 1. Drop exact duplicates
    n_before = df.shape[0]
    df = df.drop_duplicates()
    print(f"Dropped {n_before - df.shape[0]} duplicate rows. New shape: {df.shape}")

    # 2. Separate features and target
    if target_col not in df.columns:
        raise ValueError(f"Target column {target_col} not found in dataframe.")
    y = df[target_col]
    X = df.drop(columns=[target_col])

    # 3. Identify column types
    numeric_cols = X.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    # treat low-cardinality numeric as categorical if appropriate
    for col in numeric_cols.copy():
        if X[col].nunique() < 10 and X[col].dtype in ['int64', 'int32']:
            categorical_cols.append(col)
            numeric_cols.remove(col)

    print("Numeric cols:", numeric_cols)
    print("Categorical cols:", categorical_cols)

    # 4. Imputation strategies
    numeric_imputer = IterativeImputer(random_state=RANDOM_STATE) if use_iterative_imputer else KNNImputer(n_neighbors=5)
    categorical_imputer = SimpleImputer(strategy='most_frequent')

    # 5. Encoding strategies
    low_card_cat = [c for c in categorical_cols if X[c].nunique() <= max_onehot_cardinality]
    high_card_cat = [c for c in categorical_cols if X[c].nunique() > max_onehot_cardinality]

    print(f"OneHot for {len(low_card_cat)} cols, TargetEncoding for {len(high_card_cat)} cols")

    # Column transformers
    transformers = []

    if numeric_cols:
        transformers.append(('num', Pipeline([('imputer', numeric_imputer), ('scaler', StandardScaler())]), numeric_cols))

    if low_card_cat:
        transformers.append(('ohe', Pipeline([('imputer', categorical_imputer),
                                             ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))]), low_card_cat))

    # High-cardinality: target encoding (only for supervised tasks)
    if high_card_cat:
        # target encoder requires y; we will apply it separately
        pass  # handled later

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop', sparse_threshold=0)

    # Fit-transform numeric + low-cardinality categorical
    X_processed = preprocessor.fit_transform(X, y)
    # Build feature names for transformed columns
    feature_names = []
    if numeric_cols:
        feature_names += numeric_cols
    if low_card_cat:
        # get ohe names
        ohe = preprocessor.named_transformers_['ohe'].named_steps['ohe']
        ohe_names = ohe.get_feature_names_out(low_card_cat).tolist()
        feature_names += ohe_names

    X_proc_df = pd.DataFrame(X_processed, columns=feature_names, index=X.index)

    # Apply target encoding for high-cardinality categorical columns if supervised
    if high_card_cat and y is not None:
        te = ce.TargetEncoder(cols=high_card_cat)
        X_high = X[high_card_cat].copy()
        X_high = te.fit_transform(X_high, y)
        X_proc_df = pd.concat([X_proc_df, X_high], axis=1)

    return X_proc_df, y, preprocessor

# Example usage (will run if df detected)
if 'df' in globals():
    try:
        X, y, preproc = preprocess_dataframe(df, target_col)
        print("Preprocessing complete. X shape:", X.shape)
        display(X.head())
    except Exception as e:
        print("Preprocessing failed:", e)

# %%
# 6. EDA: distributions, boxplots, correlation heatmap, and automated comments
import matplotlib.pyplot as plt
import seaborn as sns

def eda_plots(df, target_col=None, max_plots=12):
    print("Running EDA...")
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    # Histograms + boxplots for numerical features (up to max_plots)
    for i, col in enumerate(numeric_cols[:max_plots]):
        fig, axes = plt.subplots(1,2, figsize=(12,4))
        sns.histplot(df[col].dropna(), ax=axes[0], kde=True)
        axes[0].set_title(f'Histogram of {col} (n={df[col].notna().sum()})')
        sns.boxplot(x=df[col], ax=axes[1])
        axes[1].set_title(f'Boxplot of {col}')
        plt.tight_layout()
        plt.show()
        # Interpretation hint
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)].shape[0]
        print(f"Interprétation: {col} a médiane {df[col].median():.3f}, presence d'outliers estimée: {outliers} observations.")

    # Correlation heatmap
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr()
        plt.figure(figsize=(10,8))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
        plt.title("Correlation matrix")
        plt.show()
        # Interpret top correlations
        corr_pairs = corr.abs().unstack().sort_values(ascending=False).drop_duplicates()
        top = corr_pairs[corr_pairs < 1].head(5)
        print("Top correlations (absolute):")
        print(top)

    # Categorical overview
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    for col in cat_cols[:min(len(cat_cols), 6)]:
        vc = df[col].value_counts(dropna=False).head(10)
        print(f"\nColonne catégorielle: {col} — top valeurs:\n{vc}")
        sns.barplot(x=vc.index.astype(str), y=vc.values)
        plt.xticks(rotation=45)
        plt.title(f"Top categories for {col}")
        plt.show()

# Run EDA on raw df
if 'df' in globals():
    eda_plots(df, target_col=target_col)

# %%
# 7. Feature engineering examples (generic)
def add_feature_engineering(df):
    df = df.copy()
    # Example 1: interaction terms between top numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_cols) >= 2:
        a, b = numeric_cols[:2]
        df[f'{a}_x_{b}'] = df[a] * df[b]
        print(f"Created interaction feature: {a}_x_{b}")
    # Example 2: count of missing per row
    df['missing_count'] = df.isna().sum(axis=1)
    # Example 3: simple datetime extraction if any datetime columns detected
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[f'{col}_year'] = df[col].dt.year
            df[f'{col}_month'] = df[col].dt.month
    return df

if 'df' in globals():
    df_fe = add_feature_engineering(df)
    print("Feature engineering done. New shape:", df_fe.shape)
    display(df_fe.head())

# %%
# 8. Modeling: auto-detect task type and run ML pipeline comparing 3 algorithms
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error, r2_score

def is_classification(y):
    # heuristics: dtype object or few unique values
    if y.dtype == 'O' or y.dtype.name == 'category' or y.dtype == 'bool':
        return True
    if y.nunique() <= 20 and y.nunique() < 0.1 * len(y):
        return True
    return False

def run_models(X, y):
    # detect task
    task = 'classification' if is_classification(y) else 'regression'
    print("Detected task:", task)
    # split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y if task=='classification' else None)

    # choose models
    if task == 'classification':
        models = {
            'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
            'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
        }
        scoring = 'f1_macro'
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    else:
        models = {
            'Ridge': Ridge(random_state=RANDOM_STATE),
            'RandomForest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
            'GradientBoosting': GradientBoostingRegressor(random_state=RANDOM_STATE)
        }
        scoring = 'neg_root_mean_squared_error'
        cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    results = {}
    for name, model in models.items():
        print(f"Evaluating {name} ...")
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        results[name] = cv_scores
        print(f"{name} CV {scoring}: mean={np.mean(cv_scores):.4f}, std={np.std(cv_scores):.4f}")

    # Hyperparameter tuning example for RandomForest and GradientBoosting
    print("\nHyperparameter tuning examples (this can be time-consuming):")
    if task == 'classification':
        rf = RandomForestClassifier(random_state=RANDOM_STATE)
        param_dist = {'n_estimators':[100,200,400], 'max_depth':[None,5,10,20], 'min_samples_split':[2,5,10]}
        search = RandomizedSearchCV(rf, param_dist, n_iter=10, cv=cv, scoring=scoring, random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X_train, y_train)
        print("RandomizedSearchCV best params (RandomForest):", search.best_params_)
        best_rf = search.best_estimator_
        # Evaluate on test set
        y_pred = best_rf.predict(X_test)
        print("Test accuracy:", accuracy_score(y_test, y_pred))
        print("Test F1 macro:", f1_score(y_test, y_pred, average='macro'))
    else:
        rf = RandomForestRegressor(random_state=RANDOM_STATE)
        param_dist = {'n_estimators':[100,200,400], 'max_depth':[None,5,10,20], 'min_samples_split':[2,5,10]}
        search = RandomizedSearchCV(rf, param_dist, n_iter=10, cv=cv, scoring=scoring, random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X_train, y_train)
        print("RandomizedSearchCV best params (RandomForest):", search.best_params_)
        best_rf = search.best_estimator_
        y_pred = best_rf.predict(X_test)
        rmse = mean_squared_error(y_test, y_pred, squared=False)
        print("Test RMSE:", rmse)
        print("Test R2:", r2_score(y_test, y_pred))

    return results, best_rf

# Run modeling if preprocessing succeeded
best_model = None
if 'X' in globals() and 'y' in globals():
    try:
        results, best_model = run_models(X, y)
    except Exception as e:
        print("Modeling failed:", e)

# %%
# 9. Feature importance and model interpretation (if tree-based model available)
def show_feature_importances(model, X, top_n=20):
    if hasattr(model, 'feature_importances_'):
        fi = model.feature_importances_
        feat = pd.Series(fi, index=X.columns).sort_values(ascending=False).head(top_n)
        print(feat)
        plt.figure(figsize=(8,6))
        sns.barplot(x=feat.values, y=feat.index)
        plt.title("Top feature importances")
        plt.show()
    else:
        print("Model has no feature_importances_. Consider using shap for detailed interpretation (install shap).")

if best_model is not None and 'X' in globals():
    try:
        show_feature_importances(best_model, X)
    except Exception as e:
        print("Could not show feature importances:", e)

# %%
# 10. Save model and preprocessor for later use
import joblib
if best_model is not None:
    os.makedirs('artifacts', exist_ok=True)
    joblib.dump(best_model, 'artifacts/best_model.joblib')
    if 'preproc' in globals():
        joblib.dump(preproc, 'artifacts/preprocessor.joblib')
    print("Saved model and preprocessor to artifacts/")

# %%
# 11. Short summary & next steps (printed)
print("""
Summary of this automated pipeline:
- Downloads dataset from Kaggle (if dataset URL/ref provided and kaggle credentials available)
- Loads first CSV file found (you may set FORCE_CSV_FILENAME to choose a different file)
- Attempts to auto-detect target column (override with TARGET_COLUMN if needed)
- Performs preprocessing: deduplication, iterative imputation for numeric, frequent imputation for categoricals,
  one-hot encoding for low-cardinality categoricals and target encoding for high-cardinality ones, scaling.
- Runs EDA with plots and brief textual interpretations
- Performs modeling: compares 3 algorithms (classification/regression auto-detected) using cross-validation,
  runs RandomizedSearchCV on RandomForest as an example of hyperparameter tuning.
- Saves best model and preprocessor in artifacts/

Next steps / tips:
- Edit TARGET_COLUMN to ensure the correct column is used as target.
- For large datasets, consider sampling or increasing compute resources.
- Add more feature engineering tailored to your domain.
- For classification with class imbalance, add resampling or class_weight strategies.
""")
# End of script

